# Classical Signal Processing & Control Systems
This notebook covers mathematical modeling and feedback loop control for mechanical and electrical systems.

## 1. [Mass-spring-damper model](https://en.wikipedia.org/wiki/Mass-spring-damper_model)
![](images/Mass_spring_damper.png)

Following Newton's Second low ($\Sigma F = m.a$):
$$m \ddot{x}(t) + b \dot{x}(t) + k x(t) = F(t)$$
Where $x(t), \dot{x}(t), \ddot{x}(t)$ are respectively is the mass position, velocity and acceleration over time, $m$ its mass, $b$ is the damping coefficient, $k$ is the spring stiffness constant, and $F(t)$ is an external applied force.

To analyze the system's natural behavior, we rewrite this into standard second-order form:

$$\ddot{x}(t) + 2\zeta\omega_0 \dot{x}(t) + \omega_0^2 x(t) = \frac{F(t)}{m}$$

Where:

Natural Undamped Angular Frequency: $\omega_0 = \sqrt{\frac{k}{m}}$

Damping Ratio Parameter: $\zeta = \frac{b}{2\sqrt{km}}$

**Goal:**
Control mass linear position $x(t)$ by applying an external force $F(t)$.

**Steps:**
1. Understand `kp`, `ki`, `kd` influence by reading [PID controller in control system](https://www.geeksforgeeks.org/electronics-engineering/proportional-integral-derivative-controller-in-control-system/).
2. Set `ki`, `kd` to 0. Tune `kp` until the system oscillates around the target value.
3. Change `m` parameter, what's its influence ?
4. Change `k` parameter, what's its influence ?
5. Change `b` parameter, what's its influence ?
6. Tune `ki` and `kd` to stabilize the system and reduce the residual error.
7. Bonus: Tune PID controller using [Ziegler-Nichols method](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method).

In [ ]:
from Systems import MassSpringDamper
from Controllers import PIDController
import matplotlib.pyplot as plt

time_step = 0.01 # seconds
# TODO: default values
# m=2.0, k=10.0, b=1.5
# Same scale value: kp=45
# Tuning with acceptable overshoot: kp=45.0, ki=50.0, kd=15.0
# Tuning with no overshoot (but undershoot): kp=45.0, ki=50.0, kd=30.0
sys = MassSpringDamper(m=2.0, k=10.0, b=1.5)
pid = PIDController(kp=45.0, ki=50.0, kd=15.0)

history = []
target = 5.0
measured = 0.0

for _ in range(300):
    control_force = pid.compute(target, measured, dt=time_step)
    measured = sys.step(control_force, dt=time_step)
    history.append(measured)

plt.plot(history, label="System Trajectory output")
plt.axhline(target, color="r", linestyle="--", label="Target Goal Position")
plt.legend()
plt.title("Mass-Spring Position Regulation Loop via PID Control")
plt.show()

## 2. [DC Motor](https://en.wikipedia.org/wiki/DC_motor) Position Loop Control

By coupling Kirchhoff's Voltage Law (electrical domain) with Newton's Second Law for Rotation (mechanical domain), a DC motor is modeled by two interconnected equations:

$$\begin{aligned}
V(t) &= R i(t) + L \frac{di(t)}{dt} + K_e \dot{\theta}(t) \\
T(t) &= K_t i(t) = J \ddot{\theta}(t) + b \dot{\theta}(t)
\end{aligned}$$

Term-by-Term Breakdown
- $V(t)$ [Input Voltage]: The external control voltage applied to the motor windings.
- $R i(t)$ [Resistive Loss]: Voltage dropped as heat due to internal coil resistance ($R$).
- $L \frac{di(t)}{dt}$ [Inductive Lag]: Voltage required to overcome the armature inductance ($L$) when current changes.
- $K_e \dot{\theta}(t)$ [Back-EMF]: Opposing voltage generated by the motor spinning through its own magnetic field, proportional to angular velocity ($\dot{\theta}$).
- $K_t i(t)$ [Electromagnetic Torque]: The mechanical torque ($T$) generated proportionally by the electrical current ($i$).
- $J \ddot{\theta}(t)$ [Inertial Load]: Torque required to accelerate the rotor and attached mass inertia ($J$).
- $b \dot{\theta}(t)$ [Viscous Friction]: Torque lost to mechanical drag and damping ($b$) proportional to velocity.2.

Second-Order System RepresentationWhen armature inductance ($L$) is small enough to be neglected ($L \approx 0$), the equations combine directly into a standard second-order differential system mapping input voltage directly to angular position:$$\ddot{\theta}(t) + \left( \frac{b \cdot R + K_t \cdot K_e}{J \cdot R} \right) \dot{\theta}(t) = \left( \frac{K_t}{J \cdot R} \right) V(t)$$Rewritten in the standard second-order form:$$\ddot{\theta}(t) + 2\zeta\omega_0 \dot{\theta}(t) = K_{sys}\omega_0^2 V(t)$$Where:Natural Undamped Frequency: $\omega_0 = \sqrt{\frac{K_t K_e}{J R}}$ (assuming low mechanical friction $b$)Damping Ratio: $\zeta = \frac{b R + K_t K_e}{2 \sqrt{J R K_t K_e}}$

**Goal:**
Control rotor angular position $\theta(t)$ by applying input voltage $V(t)$.

**Steps:**
1. Understand `kp`, `ki`, `kd` influence by reading [PID controller in control system](https://www.geeksforgeeks.org/electronics-engineering/proportional-integral-derivative-controller-in-control-system/).
2. Set `ki`, `kd` to 0. Tune `kp` until the system oscillates around the target value.
3. Change `R` parameter, what's its influence ?
4. Change `J` parameter, what's its influence ?
5. Change `b` parameter, what's its influence ?
6. Change `Kt` parameter, what's its influence ?
7. Change `Ke` parameter, what's its influence ?
8. Tune `ki` and `kd` to stabilize the system and reduce the residual error.
9. Bonus: Tune PID controller using [Ziegler-Nichols method](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method).


In [ ]:
from Systems.DCMotorPositionSystem import DCMotorPositionSystem
from Controllers import PIDController
import matplotlib.pyplot as plt

time_step = 0.01 # seconds
# TODO: base configuration
# (R=2.0, L=0.5, J=0.02, b=0.1, Kt=0.01, Ke=0.01
# Same magnitude: kp=45
# Correct control: kp=45.0, ki=0.0, kd=15.0
sys = DCMotorPositionSystem(R=2.0, L=0.5, J=0.02, b=0.1, Kt=0.01, Ke=0.01)
pid = PIDController(kp=45.0, ki=0.0, kd=15.0)

history = []
target = 5.0
measured = 0.0

for _ in range(300):
    control_force = pid.compute(target, measured, dt=time_step)
    measured = sys.step(control_force, dt=time_step)
    history.append(measured)

plt.plot(history, label="System Trajectory output")
plt.axhline(target, color="r", linestyle="--", label="Target Goal Position")
plt.legend()
plt.title("Mass-Spring Position Regulation Loop via PID Control")
plt.show()